# 6.1.6 Context Length Extension

Compare RoPE vs sinusoidal positional encodings and visualise attention patterns for long sequences.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

def sinusoidal_pe(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div)
    return pe

def rope_pe(seq_len, d_model):
    freqs = 1.0 / (10000 ** (np.arange(0, d_model, 2) / d_model))
    angles = np.outer(np.arange(seq_len), freqs)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.cos(angles)
    pe[:, 1::2] = np.sin(angles)
    return pe

seq_len, d_model = 256, 64
sin = sinusoidal_pe(seq_len, d_model)
rope = rope_pe(seq_len, d_model)
print('Sinusoidal PE:', sin.shape, '  RoPE:', rope.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(sin[:64, :32].T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set(title='Sinusoidal PE', xlabel='Position', ylabel='Dim')
axes[1].imshow(rope[:64, :32].T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[1].set(title='RoPE', xlabel='Position', ylabel='Dim')
plt.tight_layout()
plt.savefig('output/context_length.png')
print('Saved context_length.png')

In [ ]:
sim_sin = [np.dot(sin[0], sin[i]) / (np.linalg.norm(sin[0]) * np.linalg.norm(sin[i]) + 1e-8) for i in range(seq_len)]
sim_rope = [np.dot(rope[0], rope[i]) / (np.linalg.norm(rope[0]) * np.linalg.norm(rope[i]) + 1e-8) for i in range(seq_len)]
plt.figure(figsize=(8, 4))
plt.plot(sim_sin, label='Sinusoidal', color='steelblue')
plt.plot(sim_rope, label='RoPE', color='tomato')
plt.xlabel('Position'); plt.ylabel('Cosine Similarity to pos 0')
plt.title('Position Similarity Decay'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/pe_similarity.png')
print('Saved pe_similarity.png')